# 01 - Data Understanding (Give Me Some Credit)

**Goal:** understand the structure of the data, define the target, and list the anomalies that need
a decision later. This notebook runs on the **full raw file** and is deliberately **target-free**:
it only looks at structure, counts and distributions, never at how the target relates to a variable.
All target-aware exploration happens after the split, on the development set only (notebooks 02-03).

**Target definition:** `SeriousDlqin2yrs = 1` means the borrower was 90+ days past due within the
2 years after the observation date (24-month performance window).

In [1]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

TARGET = "SeriousDlqin2yrs"
LATE = ["NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
        "NumberOfTime60-89DaysPastDueNotWorse"]

df = pd.read_csv("../data/cs-training.csv", index_col=0)
df.index.name = "Id"

## 1. Structure

In [2]:
print("Shape:", df.shape)
df.dtypes

Shape: (150000, 11)


SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64
dtype: object

## 2. Target and class balance

In [3]:
n_bad = int(df[TARGET].sum())
n_good = len(df) - n_bad
print(f"Bad = {n_bad:,} ({df[TARGET].mean():.2%}) | Good = {n_good:,} | "
      f"odds good:bad = {n_good / n_bad:.1f}:1")

Bad = 10,026 (6.68%) | Good = 139,974 | odds good:bad = 14.0:1


The portfolio is imbalanced (about 6.7% bad) but there are 10k+ bad cases, enough for binning.
We will **not** use SMOTE or class weights, because they distort the predicted probabilities (PD).

## 3. Missing values

In [4]:
miss = df.isna().sum().to_frame("n_missing")
miss["pct"] = (miss["n_missing"] / len(df) * 100).round(2)
miss[miss.n_missing > 0]

,n_missing,pct
MonthlyIncome,29731,19.82
NumberOfDependents,3924,2.62


In [5]:
dep_na = df.NumberOfDependents.isna()
inc_na = df.MonthlyIncome.isna()
print("Rows missing dependents:", dep_na.sum())
print("...of which also missing income:", (dep_na & inc_na).sum())
print("Every row missing dependents is also missing income:", bool(dep_na.sum() == (dep_na & inc_na).sum()))

Rows missing dependents: 3924
...of which also missing income: 3924
Every row missing dependents is also missing income: True


## 4. Descriptive statistics

In [6]:
df.describe(percentiles=[.01, .05, .5, .95, .99]).T.round(3)

,count,mean,std,min,1%,5%,50%,95%,99%,max
SeriousDlqin2yrs,150000.0,0.067,0.250,0.0,0.0,0.000,0.000,1.0,1.000,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048,249.755,0.0,0.0,0.000,0.154,1.0,1.093,50708.0
age,150000.0,52.295,14.772,0.0,24.0,29.000,52.000,78.0,87.000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421,4.193,0.0,0.0,0.000,0.000,2.0,4.000,98.0
DebtRatio,150000.0,353.005,2037.819,0.0,0.0,0.004,0.367,2449.0,4979.040,329664.0
MonthlyIncome,120269.0,6670.221,14384.674,0.0,0.0,1300.000,5400.000,14587.6,25000.000,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.453,5.146,0.0,0.0,2.000,8.000,18.0,24.000,58.0
NumberOfTimes90DaysLate,150000.0,0.266,4.169,0.0,0.0,0.000,0.000,1.0,3.000,98.0
NumberRealEstateLoansOrLines,150000.0,1.018,1.130,0.0,0.0,0.000,1.000,3.0,4.000,54.0
NumberOfTime60-89DaysPastDueNotWorse,150000.0,0.240,4.155,0.0,0.0,0.000,0.000,1.0,2.000,98.0


## 5. Special values 96 / 98 in the delinquency-count columns

In [7]:
sp = df[LATE].isin([96, 98])
print("Cells with 96/98 per column:", sp.sum().to_dict())
print("Rows with 96/98 in AT LEAST one column:", sp.any(axis=1).sum(),
      "| in ALL three columns:", sp.all(axis=1).sum())

Cells with 96/98 per column: {'NumberOfTime30-59DaysPastDueNotWorse': 269, 'NumberOfTimes90DaysLate': 269, 'NumberOfTime60-89DaysPastDueNotWorse': 269}
Rows with 96/98 in AT LEAST one column: 269 | in ALL three columns: 269


The same 269 rows carry 96/98 in all three columns (807 cells). These look like special codes, not real counts.

## 6. Other anomalies (counts only)

In [8]:
print("Age: min =", df.age.min(), "| age<21:", (df.age < 21).sum(), "| age>=100:", (df.age >= 100).sum())
u = df.RevolvingUtilizationOfUnsecuredLines
print(f"Utilization > 1: {(u > 1).sum():,} | >= 10: {(u >= 10).sum():,}")
dr = df.DebtRatio
print(f"DebtRatio > 1: {(dr > 1).sum():,} | >= 100: {(dr >= 100).sum():,} "
      f"| of which income missing: {((dr > 1) & inc_na).sum():,}")
print("MonthlyIncome == 0:", (df.MonthlyIncome == 0).sum())
print("Fully duplicated rows:", df.duplicated().sum())

Age: min = 0 | age<21: 1 | age>=100: 13
Utilization > 1: 3,321 | >= 10: 241
DebtRatio > 1: 35,137 | >= 100: 24,402 | of which income missing: 27,904
MonthlyIncome == 0: 1634


Fully duplicated rows: 609


## Summary - questions to answer in notebook 03 (on the development set)

1. `MonthlyIncome` (19.8%) and `NumberOfDependents` (2.6%) are missing.
2. 269 rows with 96/98 codes in the delinquency-count columns.
3. `DebtRatio` has huge values, mostly where income is missing or tiny (looks like an absolute amount, not a ratio).
4. `RevolvingUtilization` has values far above 1.
5. One row with age 0; 609 duplicated rows.

Next: notebook 02 - split the raw data into development / hold-out **before** any target-aware work.